In [0]:
from pyspark.sql.types import StructType,StructField, StringType, IntegerType,FloatType, DateType, TimestampType
from pyspark.sql import functions as F

In [0]:
#create your data frame iwth brands.csv from the source_data/raw folder
catalog_name = "ecommerce"

brand_schema = StructType([
    StructField("brand_code", StringType(), True),
    StructField("brand_name", StringType(), True),
    StructField("category_code", StringType(), True)
])

df = spark.read.schema(brand_schema).csv("/Volumes/ecommerce/source_data/raw/brands.csv", header=True)
df.show()


+----------+--------------+-------------+
|brand_code|    brand_name|category_code|
+----------+--------------+-------------+
|      ACME|      AcmeTech|           CE|
|      NOVW|     NovaWave |           CE|
|      ZNTH|        Zenith|           CE|
|      BYTM|       ByteMax|           CE|
|      ECOT|       EcoTone|           CE|
|      SKYL|       SkyLink|           CE|
|     VOLT@|      VoltEdge|           CE|
|      PHTX|      Photonix|           CE|
|      URTL|    UrbanTrail|          APP|
|      COTC|    CottonClub|          APP|
|      FTFX|       FitFlex|          APP|
|      ABLF|     AmberLeaf|          APP|
|      MOSA|       Mosaic |          APP|
|      NTHR|  NorthThread |          APP|
|      CBLT|    CobaltWear|          APP|
|     CKMT@|      CookMate|          HNK|
|      HMNS|      HomeNest|          HNK|
|      PRLV|    PureLiving|          HNK|
|     CSNV@|      CasaNova|          HNK|
|      AQPR|      AquaPure|          HNK|
+----------+--------------+-------

In [0]:
#add the metadata columns for audits
df = df.withColumn("_source_file",F.col("_metadata.file_path")).withColumn("ingested_at", F.current_timestamp())
display(df)

brand_code,brand_name,category_code,_source_file,ingested_at
ACME,AcmeTech,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands.csv,2026-11-16T14:04:31.151Z
NOVW,NovaWave,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands.csv,2026-11-16T14:04:31.151Z
ZNTH,Zenith,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands.csv,2026-11-16T14:04:31.151Z
BYTM,ByteMax,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands.csv,2026-11-16T14:04:31.151Z
ECOT,EcoTone,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands.csv,2026-11-16T14:04:31.151Z
SKYL,SkyLink,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands.csv,2026-11-16T14:04:31.151Z
VOLT@,VoltEdge,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands.csv,2026-11-16T14:04:31.151Z
PHTX,Photonix,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands.csv,2026-11-16T14:04:31.151Z
URTL,UrbanTrail,APP,dbfs:/Volumes/ecommerce/source_data/raw/brands.csv,2026-11-16T14:04:31.151Z
COTC,CottonClub,APP,dbfs:/Volumes/ecommerce/source_data/raw/brands.csv,2026-11-16T14:04:31.151Z


In [0]:
#write the dataframe into the bronze table
df.write.format("delta").mode("overwrite").option("mergeSchema","true").saveAsTable(f"{catalog_name}.bronze.brz_brands")

In [0]:
from pyspark.sql.types import StructType,StructField, StringType, IntegerType,FloatType, DateType, TimestampType
from pyspark.sql import functions as F
#create df for categories from categories.csv
category_schema = StructType([
    StructField("category_code", StringType(), False),
    StructField("category_name", StringType(), True)
])
df = spark.read.schema(category_schema).csv("/Volumes/ecommerce/source_data/raw/category.csv", header=True)
df = df.withColumn("_source_file",F.col("_metadata.file_path")).withColumn("ingested_at", F.current_timestamp())
display(df)


category_code,category_name,_source_file,ingested_at
ce,Electronics,dbfs:/Volumes/ecommerce/source_data/raw/category.csv,2026-11-16T13:05:12.083Z
app,Apparel,dbfs:/Volumes/ecommerce/source_data/raw/category.csv,2026-11-16T13:05:12.083Z
hnk,Home & Kitchen,dbfs:/Volumes/ecommerce/source_data/raw/category.csv,2026-11-16T13:05:12.083Z
bpc,Beauty & Personal Care,dbfs:/Volumes/ecommerce/source_data/raw/category.csv,2026-11-16T13:05:12.083Z
bks,Books,dbfs:/Volumes/ecommerce/source_data/raw/category.csv,2026-11-16T13:05:12.083Z
grcy,Grocery,dbfs:/Volumes/ecommerce/source_data/raw/category.csv,2026-11-16T13:05:12.083Z
toy,Toys & Games,dbfs:/Volumes/ecommerce/source_data/raw/category.csv,2026-11-16T13:05:12.083Z
spt,Sports & Outdoors,dbfs:/Volumes/ecommerce/source_data/raw/category.csv,2026-11-16T13:05:12.083Z
app,Apparel,dbfs:/Volumes/ecommerce/source_data/raw/category.csv,2026-11-16T13:05:12.083Z
grcy,Grocery,dbfs:/Volumes/ecommerce/source_data/raw/category.csv,2026-11-16T13:05:12.083Z


In [0]:
#write it into a delta table in bronze schema
catalog_name = "ecommerce"
df.write.format("delta").mode("overwrite").option("mergeSchema","true").saveAsTable(f"{catalog_name}.bronze.brz_categories")

In [0]:
#make a dataframe for the products csv
product_schema = StructType([
    StructField("product_id", StringType(), False),
    StructField("sku", StringType(), True),
    StructField("category_code", StringType(), True),
    StructField("brand_code", StringType(), True),
    StructField("color", StringType(), True),
    StructField("size", StringType(), True),
    StructField("material", StringType(), True),
    StructField("weight_grams", StringType(), True),
    StructField("length_cm", StringType(), True),
    StructField("width_cm", FloatType(), True),
    StructField("height_cm", FloatType(), True),
    StructField("rating_count", IntegerType(), True),
    StructField("file_name", StringType(), True),
    StructField("ingest_timestamp", TimestampType(), True)
])
df = spark.read.schema(product_schema).csv("/Volumes/ecommerce/source_data/raw/products.csv",header=True)
df = df.withColumn("_source_file",F.col("_metadata.file_path")).withColumn("ingested_at", F.current_timestamp())
display(df)


product_id,sku,category_code,brand_code,color,size,material,weight_grams,length_cm,width_cm,height_cm,rating_count,file_name,ingest_timestamp,_source_file,ingested_at
2000000000015,STCR-HNK-00001,hnk,stcr,White,One-Size,Coton,305g,"22,2",17.1,6.3,0,null,null,dbfs:/Volumes/ecommerce/source_data/raw/products.csv,2026-11-16T13:14:24.952Z
2000000000022,HMNS-HNK-00002,hnk,hmns,Silver,One-Size,Steel,682g,"18,2",12.3,3.7,1,null,null,dbfs:/Volumes/ecommerce/source_data/raw/products.csv,2026-11-16T13:14:24.952Z
2000000000039,NOVW-CE-00003,ce,novw,Purple,One-Size,Wood,243g,"18,2",13.9,4.2,0,null,null,dbfs:/Volumes/ecommerce/source_data/raw/products.csv,2026-11-16T13:14:24.952Z
2000000000046,URTL-APP-00004,app,urtl,Silver,S,Ruber,225g,"17,6",4.6,5.8,50,null,null,dbfs:/Volumes/ecommerce/source_data/raw/products.csv,2026-11-16T13:14:24.952Z
2000000000053,GGRN-GRC-00005,grcy,ggrn,Silver,One-Size,Ruber,455g,"27,2",15.8,7.4,-4,null,null,dbfs:/Volumes/ecommerce/source_data/raw/products.csv,2026-11-16T13:14:24.952Z
2000000000060,SLKE-BPC-00006,bpc,slke,Purple,One-Size,Plastic,232g,"28,0",13.8,6.1,0,null,null,dbfs:/Volumes/ecommerce/source_data/raw/products.csv,2026-11-16T13:14:24.952Z
2000000000077,VOLT-CE-00007,ce,volt,Blue,One-Size,Plastic,507g,"27,2",12.1,6.4,5,null,null,dbfs:/Volumes/ecommerce/source_data/raw/products.csv,2026-11-16T13:14:24.952Z
2000000000084,CBLT-APP-00008,app,cblt,Blue,XS,Polyester,261g,"27,7",8.5,7.0,0,null,null,dbfs:/Volumes/ecommerce/source_data/raw/products.csv,2026-11-16T13:14:24.952Z
2000000000091,ARFT-SPT-00009,spt,arft,Blue,XL,Plastic,59g,"12,5",19.0,7.9,11,null,null,dbfs:/Volumes/ecommerce/source_data/raw/products.csv,2026-11-16T13:14:24.952Z
2000000000107,MOSA-APP-0000A,app,mosa,White,L,Polyester,238g,"10,7",17.7,10.3,6,null,null,dbfs:/Volumes/ecommerce/source_data/raw/products.csv,2026-11-16T13:14:24.952Z


In [0]:
#write it into a delta table in bronze schema
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog_name}.bronze.brz_products")

In [0]:
#create df for customers.csv
from pyspark.sql.types import StructType,StructField, StringType, IntegerType,FloatType, DateType, TimestampType
from pyspark.sql import functions as F
customer_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("phone", StringType(), True),
    StructField("country_code", StringType(), True),
    StructField("country", StringType(), True),
    StructField("state", StringType(), True)
])
df = spark.read.schema(customer_schema).csv("/Volumes/ecommerce/source_data/raw/customers.csv", header=True)
df = df.withColumn("_source_file",F.col("_metadata.file_path")).withColumn("ingested_at", F.current_timestamp())

#write it into a delta table in bronze schema
catalog_name = "ecommerce"
df.write.format("delta").mode("overwrite").option("mergeSchema","true").saveAsTable(f"{catalog_name}.bronze.brz_customers")
display(df)


customer_id,phone,country_code,country,state,_source_file,ingested_at
CUST000000000001,917280033536.0,IN,India,MH,dbfs:/Volumes/ecommerce/source_data/raw/customers.csv,2026-11-16T13:50:32.548Z
CUST000000000002,619489725433.0,AU,Australia,VIC,dbfs:/Volumes/ecommerce/source_data/raw/customers.csv,2026-11-16T13:50:32.548Z
CUST000000000003,919390066524.0,IN,India,TN,dbfs:/Volumes/ecommerce/source_data/raw/customers.csv,2026-11-16T13:50:32.548Z
CUST000000000004,917073741793.0,IN,India,TN,dbfs:/Volumes/ecommerce/source_data/raw/customers.csv,2026-11-16T13:50:32.548Z
CUST000000000005,618478772532.0,AU,Australia,WA,dbfs:/Volumes/ecommerce/source_data/raw/customers.csv,2026-11-16T13:50:32.548Z
CUST000000000006,916441718520.0,IN,India,GJ,dbfs:/Volumes/ecommerce/source_data/raw/customers.csv,2026-11-16T13:50:32.548Z
CUST000000000007,null,IN,India,MH,dbfs:/Volumes/ecommerce/source_data/raw/customers.csv,2026-11-16T13:50:32.548Z
CUST000000000008,446806361276.0,GB,United Kingdom,ENG,dbfs:/Volumes/ecommerce/source_data/raw/customers.csv,2026-11-16T13:50:32.548Z
CUST000000000009,18191801729.0,US,United States,MA,dbfs:/Volumes/ecommerce/source_data/raw/customers.csv,2026-11-16T13:50:32.548Z
CUST000000000010,null,IN,India,RJ,dbfs:/Volumes/ecommerce/source_data/raw/customers.csv,2026-11-16T13:50:32.548Z


In [0]:
#date.csv into a df and then into az delta table in the bronze schema
date_schema = StructType([
    StructField("date", StringType(), True),
    StructField("year", IntegerType(), True),
    StructField("day_name", StringType(), True),
    StructField("quarter", IntegerType(), True),
    StructField("week_of_year", IntegerType(), True)
])
df = spark.read.schema(date_schema).csv("/Volumes/ecommerce/source_data/raw/date.csv", header=True)
df = df.withColumn("_source_file",F.col("_metadata.file_path")).withColumn("ingested_at", F.current_timestamp())
#write this into a delta table on bronze schema
df.write.format("delta").mode("overwrite").option("mergeSchema","true").saveAsTable(f"{catalog_name}.bronze.brz_date")
display(df)


date,year,day_name,quarter,week_of_year,_source_file,ingested_at
01-08-2025,2025,friday,3,-31,dbfs:/Volumes/ecommerce/source_data/raw/date.csv,2026-11-16T13:55:26.434Z
02-08-2025,2025,SATURDAY,3,-31,dbfs:/Volumes/ecommerce/source_data/raw/date.csv,2026-11-16T13:55:26.434Z
03-08-2025,2025,SUNDAY,3,-31,dbfs:/Volumes/ecommerce/source_data/raw/date.csv,2026-11-16T13:55:26.434Z
04-08-2025,2025,MONDAY,3,-32,dbfs:/Volumes/ecommerce/source_data/raw/date.csv,2026-11-16T13:55:26.434Z
05-08-2025,2025,TUESDAY,3,-32,dbfs:/Volumes/ecommerce/source_data/raw/date.csv,2026-11-16T13:55:26.434Z
06-08-2025,2025,WEDNESDAY,3,-32,dbfs:/Volumes/ecommerce/source_data/raw/date.csv,2026-11-16T13:55:26.434Z
07-08-2025,2025,thursday,3,-32,dbfs:/Volumes/ecommerce/source_data/raw/date.csv,2026-11-16T13:55:26.434Z
08-08-2025,2025,friday,3,-32,dbfs:/Volumes/ecommerce/source_data/raw/date.csv,2026-11-16T13:55:26.434Z
09-08-2025,2025,SATURDAY,3,-32,dbfs:/Volumes/ecommerce/source_data/raw/date.csv,2026-11-16T13:55:26.434Z
10-08-2025,2025,SUNDAY,3,-32,dbfs:/Volumes/ecommerce/source_data/raw/date.csv,2026-11-16T13:55:26.434Z
